In [5]:
from fastapi import FastAPI, HTTPException, Header, Depends, Request
from pydantic import BaseModel
from dotenv import load_dotenv
import os

import requests
from typing import List, Dict, Optional
from src.utils.llmp_utils import llmp_call

class GenerateRequest(BaseModel):
    model: str
    system_prompt: str = ''
    prompt: str
    format: Optional[dict] = None
    image: Optional[str] = None
    tools: Optional[List[Dict]] = None
    src: str = None
    temperature: float = 0.5
    
class KBAgent:
    
    def __init__(self, tools_desc, model):
        
        
        from dotenv import load_dotenv
        from sentence_transformers import CrossEncoder
        
        load_dotenv()
        
        
        self.src = 'kb_agent'
        self.model = model
        self.llmp_url = os.getenv("LLMP_URL")
        self.llmp_password = os.getenv("LLMP_PASSWORD")
        self.tools_desc = tools_desc
        self.cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


        
    def llmp_call(self, prompt, system_prompt, model):
        """ 
        Call the LLMP API to generate a response
        All related to the call is processed here
        """
        
        headers = {
        "Content-Type": "application/json",
        "Authorization": self.llmp_password
    }
        
        # Construct request payload
        request_data = GenerateRequest(
        model=model,
        system_prompt=system_prompt,
        prompt=prompt,
        tools=None,
        src=self.src)
        
        payload = request_data.model_dump(exclude_none=True)

        try:
            response = requests.post(self.llmp_url, headers=headers, json=payload)
            response.raise_for_status()  # Raise an error for bad responses (4xx, 5xx)
            return response.json()
        except requests.exceptions.RequestException as e:
            print(f"Request failed: {e}")
            return None
        
    def kb_agent_response(self, user_prompt, temperature=0):
        """ 
        Encompasses logic behind tool decision making
        Calls the llmp_call method to generate a response
        """
        
        system_prompt = "You are a selector agent- Your task is to select the most appropriate knowledge base to answer the query."
        
        tools_description = "\n ".join([f"{key}: {value}" for key, value in self.tools_desc.items()])
        prompt = f"{user_prompt}\nwhich of the following knowledge bases would you use?\n {tools_description}"
        
        llmp_response = llmp_call(prompt, system_prompt, self.model, temperature,src = 'KB Agent')['message']['content']
        
        results = self.cross_encoder.predict([[llmp_response, tool] for tool in self.tools_desc.keys()])
        
        tool_scores = dict(zip(self.tools_desc.keys(), results))
        best_tool = max(tool_scores, key=tool_scores.get)
        
        
        return best_tool
        
        
    def kb_agent_chat(self, user_prompt):
            """
            Logic behind tool activation.
            Sends to agent_0_response for tool decision.
            Activates tool.

            Args:
                user_prompt (str)
            """
            
            selected_kb = self.kb_agent_response(user_prompt)

            
                
            from src.pipelines.rag_pipeline import generate_rag
                
            user_prompt_rag = user_prompt.strip('given my documents')
            rag_output = generate_rag(self.model, user_prompt,selected_kb)
            #return rag_output,user_prompt,selected_kb
            llmp_response = rag_output[0]['message']['content']
            references = rag_output[1]
                
            print(llmp_response)
                
            print("📚 References:\n")
            for doc, pages in references.items():
                print(f"📄 **{doc}**")
                print(f"   📑 Pages: {', '.join(map(str, pages))}\n")
            return llmp_response    
                
                
        

In [6]:
#from src.agent_0 import Agent0


knowledge_bases_desc = {'physics_kb':'a knowledge base with information related to physics',
              'mathematics_kb':'a knowledge base with information related to mathematics"''
              }


kb_agent = KBAgent(knowledge_bases_desc, "llama3.2:latest")

In [3]:
from src.pipelines.rag_pipeline import generate_rag

user_prompt = "what is the use of integrals in series convergence test?"
rag_output = generate_rag("llama3.2:latest", user_prompt,'mathematics_kb')

intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...


In [4]:
rag_output

({'model': 'llama3.2:latest',
  'created_at': '2025-04-09T05:43:24.6495581Z',
  'message': {'role': 'assistant',
   'content': 'The final answer is: To determine if a series converges or diverges.'},
  'done_reason': 'stop',
  'done': True,
  'total_duration': 3061642900,
  'load_duration': 1.68,
  'prompt_eval_count': 2048,
  'prompt_eval_duration': 1.08,
  'eval_count': 17,
  'eval_duration': 0.3,
  'system_prompt': 'You are an assistant. Your provide answers based on provided text.',
  'prompt': 'Based only on the following in markdown: Exponential growth and decay are covered in this chapter. 7 Techniques of Integration All the standard methods are covered but, of course, the real challenge is to be able to recognize which technique is best used in a given situation. Accordingly, in Section 7.5, Ipresent a strategy for integration. The use of computer algebra systems is discussed in Section 7.6. Here are the applications of integration—arc length and surface area—for which it is us

In [7]:
user_prompt = "what is the use of integrals in series convergence test?"
response = kb_agent.kb_agent_chat(user_prompt)

intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
The final answer is: To determine if a series converges or diverges.
📚 References:

📄 **Calculus. Stewart.pdf**
   📑 Pages: 21, 111, 113, 117



In [6]:
response

[{'id': 33,
  'text': 'Exponential growth and decay are covered in this chapter. 7 Techniques of Integration All the standard methods are covered but, of course, the real challenge is to be able to recognize which technique is best used in a given situation. Accordingly, in Section 7.5, Ipresent a strategy for integration. The use of computer algebra systems is discussed in Section 7.6. Here are the applications of integration—arc length and surface area—for which it is use- ful to have available all the techniques of integration, as well as applications to biology,economics, and physics (hydrostatic force and centers of mass). I have also included a sec-tion on probability. There are more applications here than can realistically be covered in agiven course. Instructors should select applications suitable for their students and forwhich they themselves have enthusiasm.Content 6 Inverse Functions: Exponential, Logarithmic, and Inverse Trigonometric Functions 8 Further Applications of In

In [1]:
import sys
print(sys.executable)


h:\projects\ai_based\Agent-Factory\.venv\Scripts\python.exe
